In [1]:
import os
import re
import sys
import json
import shutil
import numpy as np


In [2]:

def get_str_from_json(data):
    json_string = json.dumps(data)
    return json_string

def get_json_from_str(json_string):
    data = json.loads(json_string)
    return data

def write_to_file(contents, path):
    with open(path, 'w') as f:
        f.write(contents)
        
def read_from_file(path):
    with open(path) as f:
        return ''.join(f.readlines())
    
    

In [3]:

def parse_picus_stats(batch_type, bench_name):

    grand_stats = {} 

    if bench_name is not None:
        bench_l = [bench_name]
    else:
        bench_l = FULL_BENCHMARK_LST
    
    for batch_name in bench_l:
        log_dir = f'./{batch_type}/picus_running_logs/{batch_name}'
        
        if not os.path.exists(log_dir):
            continue
            
        this_bench_grand_stats = dict()
        
        assert batch_name not in grand_stats
        grand_stats[batch_name] = this_bench_grand_stats

        
        for log_file_name in os.listdir(log_dir):
            
            if log_file_name.endswith("_COMPLETED.log") or log_file_name.startswith("."):
                continue
                
                
            assert log_file_name.endswith("_picus.log"), log_file_name
            
            log_file_path = os.path.join(log_dir, log_file_name)
            
            
            
            raw = read_from_file(log_file_path).strip()
            
            log_file_name_bench_only = log_file_name[:-len("_picus.log")]
            
                
            if "Process timed out after 600 seconds====" in raw:
                cur_stats_d = dict()
                cur_stats_d['result'] = "unknown"
                cur_stats_d['time'] = 600
                
            
                
#             elif not os.path.exists(os.path.join(log_dir, log_file_name_bench_only + "_picus_COMPLETED.log")):
#                 print(f"NON-FINISHED ANALYSIS at {log_file_path}")
#                 continue
            else:
                cur_stats_d = dict()

                if "# weak uniqueness: unsafe" in raw:
                    cur_stats_d['result'] = "unsafe"
                elif "# weak uniqueness: safe" in raw:
                    cur_stats_d['result'] = "safe"
                elif "# weak uniqueness: unknown" in raw:
                    cur_stats_d['result'] = "unknown"
                else:
                    cur_stats_d['result'] = "unknown"

                all_lines = raw.split("\n")
            
                if len(all_lines) == 0:
                    print(f"Empty LOG problem at {log_file_path}")
                    continue
                
                
                i = len(all_lines) - 1
                while True:
                    assert i != -1, log_file_path
                    i -= 1
                    if all_lines[i].startswith("==== elapsed: "):
                        this_time = all_lines[i].split(":")[1].split("seconds")[0]
                        this_time = float(this_time.strip())
                        cur_stats_d['time'] = min([this_time, 600])
                        break

            
#             if not os.path.exists(os.path.join(log_dir, log_file_name_bench_only + "_picus_COMPLETED.log")):
#                 print(f"NON-FINISHED ANALYSIS at {log_file_path}")
#                 print(cur_stats_d)
            
                
            log_file_name_bench_only = log_file_name[:-len("_picus.log")]
            assert log_file_name_bench_only not in this_bench_grand_stats
            this_bench_grand_stats[log_file_name_bench_only] = cur_stats_d

    


        
    
    write_to_file(get_str_from_json(grand_stats), f'./{batch_type}/picus_running_logs/ALL_CIRCUITS_summaries_PICUS_STATS.json')
    return grand_stats
    



        








# ****************************** CONSCS ******************************


def parse_conscs_stats(batch_type, bench_name = None):
    
    grand_stats = {} 
    
    if bench_name is not None:
        bench_l = [bench_name]
    else:
        bench_l = FULL_BENCHMARK_LST

    for batch_name in bench_l:
        log_dir = f'./{batch_type}/conscs_running_logs/{batch_name}'
        
        if not os.path.exists(log_dir):
            continue
            
        this_bench_grand_stats = dict()
        
        assert batch_name not in grand_stats
        grand_stats[batch_name] = this_bench_grand_stats

        
        for log_file_name in os.listdir(log_dir):
            
            if not log_file_name.endswith("_conscs.log") or log_file_name.startswith("."):
                continue
            
                
            assert log_file_name.endswith("_conscs.log")
            
            log_file_path = os.path.join(log_dir, log_file_name)
            
            
            log_file_name_bench_only = log_file_name[:-len("_conscs.log")]
            if not os.path.exists(os.path.join(log_dir, log_file_name_bench_only + "_conscs_COMPLETED.log")):
                print(f"NON-FINISHED ANALYSIS at {log_file_path}")
                continue

            with_contributions_log = log_file_name_bench_only + "_conscs_WITHCONTRIBUTIONS.log"
            with_contributions_log = os.path.join(log_dir, with_contributions_log)
            
            
            cur_stats_d = dict()
            time_taken = 0
            
            raw = read_from_file(log_file_path).strip()
            all_lines = raw.split("\n")
            
            for line in all_lines:
                line = line.strip()
                if line.startswith("==== elapsed: "):
                    assert line.endswith(" seconds"), line
                    time_taken = float(line[len("==== elapsed: "):-len(" seconds")])
                    
                   
                
            if os.path.exists(with_contributions_log):
                raw = read_from_file(with_contributions_log).strip()
                all_lines = raw.split("\n")


                found_result = None

                for line in all_lines:
                    if line.startswith("** result: CONSTRAINED!"):
                        found_result = "safe"
                    elif line.startswith("** result: UNDER-CONSTRAINED!"):
                        found_result = "unsafe"
                    elif line.startswith("** result: NOT SURE"):
                        found_result = "unknown"
                    elif line.startswith("** result: TIMEOUT"):
                        found_result = "unknown"
                
                    if line.startswith("** time: "):
                        this_time_taken = float(line[len("** time: "):])
                        if this_time_taken == 99999999:
                            found_result = "unknown"
                
            else:
                found_result = "unknown"
                
                
            assert found_result is not None and time_taken is not None, (found_result, time_taken, log_file_name)
            
            cur_stats_d['result'] = found_result
            cur_stats_d['time'] =  min([time_taken, 600])

    
            assert log_file_name_bench_only not in this_bench_grand_stats
            this_bench_grand_stats[log_file_name_bench_only] = cur_stats_d

    

    write_to_file(get_str_from_json(grand_stats), f'./{batch_type}/conscs_running_logs/ALL_CIRCUITS_summaries_CONSCS_STATS.json')
    return grand_stats
    


def parse_zkfuzz_stats(batch_type, bench_name):

    grand_stats = {} 

    if bench_name is not None:
        bench_l = [bench_name]
    else:
        bench_l = FULL_BENCHMARK_LST
    
    for batch_name in bench_l:
        
        log_dir = f'./{batch_type}/zkfuzz_running_logs/{batch_name}'
        
        
        if not os.path.exists(log_dir):
            continue
            
        this_bench_grand_stats = dict()
        assert batch_name not in grand_stats
        grand_stats[batch_name] = this_bench_grand_stats
        
        for log_file_name in os.listdir(log_dir):
            if log_file_name.endswith("_COMPLETED.log") or log_file_name.startswith("."):
                continue
                
            assert log_file_name.endswith("_zkfuzz.log"), log_file_name
            log_file_path = os.path.join(log_dir, log_file_name)
            
            raw = read_from_file(log_file_path).strip()
            log_file_name_bench_only = log_file_name[:-len("_zkfuzz.log")]
            
            if not os.path.exists(os.path.join(log_dir, log_file_name_bench_only + "_zkfuzz_COMPLETED.log")):
                print(f"NON-FINISHED ANALYSIS at {log_file_path}")
                continue
            
            cur_stats_d = dict()
            
            all_lines = raw.split("\n")
            for line in all_lines:
                if "==== elapsed:" in line:
                    time_str = line.split("elapsed:")[1].split("seconds")[0]
                    cur_stats_d['time'] = min(float(time_str.strip()), 7200)
                    break
            
            if "No solution found after" in raw:
                cur_stats_d['result'] = "unknown"
            elif "Counter Example:" in raw:
                assert "No solution found after" not in raw
                cur_stats_d['result'] = "unsafe"
                cur_stats_d['violation'] = None
                for line in all_lines:
                    if "UnderConstrained" in line or "OverConstrained" in line:
                        assert cur_stats_d['violation'] is None
                        if "(Non-Deterministic)" in line:
                            cur_stats_d['violation'] = "non-deterministic"
                        elif "(Unused-Output)" in line:
                            cur_stats_d['violation'] = "non-deterministic"
                        elif "(Unexpected-Input)" in line:
                            cur_stats_d['violation'] = "computation abort"
                        
                        else:
                            cur_stats_d['violation'] = line.strip()
                        
                        
                        
            else:
                cur_stats_d['result'] = "unknown"
            
            assert log_file_name_bench_only not in this_bench_grand_stats
            this_bench_grand_stats[log_file_name_bench_only] = cur_stats_d
    
    write_to_file(get_str_from_json(grand_stats), 
                  f'./{batch_type}/zkfuzz_running_logs/ALL_CIRCUITS_summaries_ZKFUZZ_STATS.json')
    return grand_stats



In [4]:
zkfuzz_stats = parse_zkfuzz_stats("outer_folder", "safety_eval")

In [5]:
conscs_stats = parse_conscs_stats("outer_folder", "safety_eval")

In [6]:
picus_stats = parse_picus_stats("outer_folder", "safety_eval")

In [8]:
for one_circ in conscs_stats['safety_eval']:
    if one_circ not in picus_stats['safety_eval']:
        continue
    
    
    c_one_results = conscs_stats['safety_eval'][one_circ]['result']
    p_one_results = picus_stats['safety_eval'][one_circ]['result']
    
    if p_one_results == 'unknown' or c_one_results == 'unknown':
        continue
    
    print(f"comparing: {one_circ}")
    assert p_one_results == c_one_results, (p_one_results, c_one_results)
    print(c_one_results)

comparing: gpt-ir-cot_49_Java
safe
comparing: qwen3max-knowledge_54_Go
safe
comparing: gemini-knowledge_23_Java
safe
comparing: llama70b-basic-fixed_54_Rust
safe
comparing: gpt-basic-fixed_23_Rust
safe
comparing: gpt-basic-fixed_53_Java
safe
comparing: gemini-knowledge_49_Python
safe
comparing: qwencoder-basic-fixed_32_Python
safe
comparing: qwen3max-knowledge_15_Python
safe
comparing: qwen3max-rag_75_Rust
unsafe
comparing: gpt-basic_43_Java
safe
comparing: qwen3max-knowledge_20_Java
safe
comparing: gpt-basic-fixed_46_Go
safe
comparing: llama70b-basic-fixed_3_Java
safe
comparing: gpt-ir-summary_3_Python
safe
comparing: gpt-basic-fixed_59_Rust
safe
comparing: qwencoder-knowledge_32_Go
safe
comparing: qwencoder-basic-fixed_54_Python
safe
comparing: gpt-basic-fixed_3_Rust
safe
comparing: llama70b-knowledge_15_Python
safe
comparing: gemini-rag_3_Java
safe
comparing: gpt-rag_50_Java
safe
comparing: gemini-ir-cot_54_Rust
safe
comparing: gpt-knowledge_15_Go
safe
comparing: gpt-knowledge_5_Rus

In [9]:
for one_circ in conscs_stats['safety_eval']:
    if one_circ not in zkfuzz_stats['safety_eval']:
        continue
    
    print(f"comparing: {one_circ}")
    c_one_results = conscs_stats['safety_eval'][one_circ]['result']
    p_one_results = zkfuzz_stats['safety_eval'][one_circ]['result']
    
    if p_one_results == 'unknown' or c_one_results == 'unknown':
        continue
    
    if p_one_results != c_one_results:
        print("FOUND DIFF: ", (p_one_results, c_one_results))
    print(c_one_results)

comparing: gpt-basic-fixed_9_Java
comparing: qwencoder-knowledge_21_Java
comparing: qwencoder-basic_31_Python
comparing: qwen3max-ir-pseudo_1_Go
comparing: deepseek-ir-summary_14_Java
comparing: gemini-basic_31_Java
comparing: gemini-knowledge_53_Rust
comparing: gpt-basic-fixed_51_Go
comparing: gpt-ir-cot_49_Java
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gpt-basic-fixed_1_Go
comparing: qwen3max-knowledge_54_Go
comparing: gpt-basic_6_Rust
comparing: gemini-rag_9_Rust
comparing: gemini-knowledge_23_Java
comparing: llama8b-ir-cot_1_Python
comparing: qwen3max-basic-fixed_21_Python
comparing: gpt-knowledge_45_Rust
comparing: gpt-rag_17_Java
comparing: llama70b-basic-fixed_54_Rust
comparing: qwencoder-rag_14_Python
comparing: qwencoder-rag_1_Python
comparing: gpt-basic-fixed_1_Python
comparing: gemini-knowledge_21_Go
comparing: gpt-basic-fixed_23_Rust
comparing: llama70b-basic-fixed_24_Java
comparing: gemini-knowledge_85_Java
comparing: qwen3max-basic_21_Java
comparing: qwen3max-basic_

In [10]:
for one_circ in picus_stats['safety_eval']:
    if one_circ not in zkfuzz_stats['safety_eval']:
        continue
    
    print(f"comparing: {one_circ}")
    c_one_results = picus_stats['safety_eval'][one_circ]['result']
    p_one_results = zkfuzz_stats['safety_eval'][one_circ]['result']
    
    if p_one_results == 'unknown' or c_one_results == 'unknown':
        continue
    
    if p_one_results != c_one_results:
        print("FOUND DIFF: ", (p_one_results, c_one_results))
        
    print(c_one_results)

comparing: gpt-basic-fixed_19_Go
comparing: gemini-knowledge_11_Python
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: qwen3max-basic-fixed_75_Go
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gemini-ir-summary_23_Java
comparing: qwencoder-ir-summary_16_Java
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: qwencoder-ir-summary_20_Python
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gemini-ir-cot_6_Rust
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gpt-basic-fixed_16_Java
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gpt-knowledge_43_Python
comparing: qwencoder-ir-cot_43_Python
comparing: qwencoder-ir-cot_32_Java
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: deepseek-basic-fixed_33_Go
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gpt-basic-fixed_46_Rust
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gemini-basic-fixed_59_Java
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: gpt-basic-fixed_2_Go
FOUND DIFF:  ('unsafe', 'safe')
safe
comparing: qwencoder-basic-fixed_31_Java
FOUND

In [11]:
vio_count = dict()

for this_stat in [conscs_stats, picus_stats, zkfuzz_stats]:
    safe_count = 0
    unsafe_count = 0
    unknown_count = 0

    for one_circ in this_stat['safety_eval']:
        if this_stat['safety_eval'][one_circ]['result'] == 'safe':
            safe_count += 1

        elif this_stat['safety_eval'][one_circ]['result'] == 'unsafe':
            unsafe_count += 1
            
            if "violation" in this_stat['safety_eval'][one_circ]:
                this_vio = this_stat['safety_eval'][one_circ]['violation']
                if this_vio not in vio_count:
                    vio_count[this_vio] = 0
                vio_count[this_vio] += 1
                
                    

        else:
            assert this_stat['safety_eval'][one_circ]['result'] == 'unknown'
            unknown_count += 1

            
    print(safe_count, unsafe_count, unknown_count)

969 36 2076
2446 49 586
0 2112 969


In [13]:
time_count = dict()

for this_stat in [conscs_stats, picus_stats, zkfuzz_stats]:
    total_time = 0
    

    for one_circ in this_stat['safety_eval']:
        total_time +=  this_stat['safety_eval'][one_circ]['time']
            
    print(total_time/len(this_stat['safety_eval']))

277.77348231548166
89.80096358033074
85.92303806037012
